In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [6]:
CSV_PATH = "/content/drive/MyDrive/Trained_phishing_ai.csv"
MODEL_SAVE_PATH = "/content/drive/MyDrive/phishing_ai_detector.pkl"

In [7]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Shape: (50000, 11)

Columns:
['id', 'split', 'is_phishing', 'is_ai_assisted', 'subject', 'body', 'sender_display_name', 'sender_email', 'reply_to', 'has_link', 'has_attachment']

First 5 rows:


,id,split,is_phishing,is_ai_assisted,subject,body,sender_display_name,sender_email,reply_to,has_link,has_attachment
0,E017629,train,0,1,Follow-up: Trip confirmation,"Good afternoon Mia,\n\nTo keep things moving, ...",Service Team,service.team@support.example,NaN,1,0
1,E002371,test,0,0,Receipt for your recent order — please review,"Hi Parker,\n\nWe've attached the receipt linke...",Customer Care,customer.care@vendor.example,NaN,0,1
2,E005202,train,0,0,Hotel and journey details for upcoming trip,"Good afternoon,\n\nHope you're well. Have a lo...","Operations Team, Beacon Health",operations.team.beacon.health@notice.example,NaN,1,1
3,E013503,validation,0,1,Update — Calendar change,"Good morning,\n\nWe've adjusted the calendar e...","Project Office, Beacon Health",project.office.beacon.health@mail.example,NaN,0,0
4,E010112,train,0,0,Payroll update | this month,"Dear Quinn,\n\nHope your week is going smoothl...",HR Operations,hr.operations@service.example,NaN,0,0


In [8]:
required_cols = ["split", "subject", "body", "is_phishing", "is_ai_assisted"]

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["split"] = df["split"].astype(str).str.strip().str.lower()
df["subject"] = df["subject"].fillna("").astype(str).str.strip()
df["body"] = df["body"].fillna("").astype(str).str.strip()

# Combine subject and body into one text field
df["text"] = "Subject: " + df["subject"] + "\n\nBody: " + df["body"]

print(df["split"].value_counts(dropna=False))
print("\nLabel distribution:")
print(df[["is_phishing", "is_ai_assisted"]].value_counts().sort_index())

split
train         35000
test           7500
validation     7500
Name: count, dtype: int64

Label distribution:
is_phishing  is_ai_assisted
0            0                 12500
             1                 12500
1            0                 12500
             1                 12500
Name: count, dtype: int64


In [9]:
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"].isin(["validation", "val"])].copy()
test_df  = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

if train_df.empty or test_df.empty:
    raise ValueError("Train or test split is empty. Check the 'split' column values.")

Train: (35000, 12)
Validation: (7500, 12)
Test: (7500, 12)


In [10]:
X_train = train_df["text"]
X_val   = val_df["text"] if not val_df.empty else None
X_test  = test_df["text"]

y_train = train_df[["is_phishing", "is_ai_assisted"]]
y_val   = val_df[["is_phishing", "is_ai_assisted"]] if not val_df.empty else None
y_test  = test_df[["is_phishing", "is_ai_assisted"]]

In [11]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        min_df=2,
        strip_accents="unicode"
    )),
    ("clf", MultiOutputClassifier(
        LogisticRegression(max_iter=1000)
    ))
])

print(model)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=30000, min_df=2,
                                 ngram_range=(1, 2), strip_accents='unicode')),
                ('clf',
                 MultiOutputClassifier(estimator=LogisticRegression(max_iter=1000)))])


In [12]:
model.fit(X_train, y_train)
print("Training completed.")

Training completed.


In [13]:
pred = model.predict(X_test)

print("========== PHISHING DETECTION ==========")
print(classification_report(y_test["is_phishing"], pred[:, 0], digits=4))

print("========== AI-ASSISTED DETECTION ==========")
print(classification_report(y_test["is_ai_assisted"], pred[:, 1], digits=4))

exact_match = ((pred == y_test.values).all(axis=1)).mean()
print(f"Exact-match accuracy (both labels correct): {exact_match:.4f}")

========== PHISHING DETECTION ==========
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      3750
           1     1.0000    1.0000    1.0000      3750

    accuracy                         1.0000      7500
   macro avg     1.0000    1.0000    1.0000      7500
weighted avg     1.0000    1.0000    1.0000      7500

========== AI-ASSISTED DETECTION ==========
              precision    recall  f1-score   support

           0     1.0000    0.9973    0.9987      3750
           1     0.9973    1.0000    0.9987      3750

    accuracy                         0.9987      7500
   macro avg     0.9987    0.9987    0.9987      7500
weighted avg     0.9987    0.9987    0.9987      7500

Exact-match accuracy (both labels correct): 0.9987


In [14]:
if X_val is not None and y_val is not None and len(X_val) > 0:
    val_pred = model.predict(X_val)

    print("========== VALIDATION: PHISHING ==========")
    print(classification_report(y_val["is_phishing"], val_pred[:, 0], digits=4))

    print("========== VALIDATION: AI-ASSISTED ==========")
    print(classification_report(y_val["is_ai_assisted"], val_pred[:, 1], digits=4))

    val_exact_match = ((val_pred == y_val.values).all(axis=1)).mean()
    print(f"Validation exact-match accuracy: {val_exact_match:.4f}")
else:
    print("No validation split found.")

========== VALIDATION: PHISHING ==========
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      3750
           1     1.0000    1.0000    1.0000      3750

    accuracy                         1.0000      7500
   macro avg     1.0000    1.0000    1.0000      7500
weighted avg     1.0000    1.0000    1.0000      7500

========== VALIDATION: AI-ASSISTED ==========
              precision    recall  f1-score   support

           0     1.0000    0.9997    0.9999      3750
           1     0.9997    1.0000    0.9999      3750

    accuracy                         0.9999      7500
   macro avg     0.9999    0.9999    0.9999      7500
weighted avg     0.9999    0.9999    0.9999      7500

Validation exact-match accuracy: 0.9999


In [15]:
joblib.dump(model, MODEL_SAVE_PATH)
print(f"Model saved to: {MODEL_SAVE_PATH}")

Model saved to: /content/drive/MyDrive/phishing_ai_detector.pkl


In [16]:
def predict_email(subject, body):
    text = f"Subject: {subject}\n\nBody: {body}"
    pred = model.predict([text])[0]

    phishing_label = "Phishing" if pred[0] == 1 else "Normal"
    ai_label = "AI-generated/AI-assisted" if pred[1] == 1 else "Human-written"

    return {
        "is_phishing": int(pred[0]),
        "is_ai_assisted": int(pred[1]),
        "phishing_result": phishing_label,
        "ai_result": ai_label
    }

# Example test
result = predict_email(
    "Urgent: Verify your account now",
    "We detected suspicious activity on your account. Click the secure link below to avoid suspension."
)

print(result)

{'is_phishing': 1, 'is_ai_assisted': 1, 'phishing_result': 'Phishing', 'ai_result': 'AI-generated/AI-assisted'}
